# Building a Neural Network from Scratch — Day 3

Your `NeuralNetwork` class is complete. Today we use it to run three experiments and think carefully about what happens when a network has too few parameters, too many, or roughly the right number.

If you are still finishing Day 2 parts, that is fine — work through this and come back to tighten earlier cells when there is time.

---

## Setup

Paste your `NeuralNetwork` class in the cell below, or import it. The helpers provided here are all you will need today.

In [ ]:
# Your NeuralNetwork class here (or import it).

import random
import math
import matplotlib.pyplot as plt

In [ ]:
# Provided: data loading.

def load_mnist_csv(filepath, max_rows=None):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or not line[0].isdigit():
                continue
            parts = line.split(',')
            label = int(parts[0])
            pixels = [int(p) / 255.0 for p in parts[1:]]
            if len(pixels) == 784:
                data.append((label, pixels))
            if max_rows and len(data) >= max_rows:
                break
    return data

In [ ]:
# Provided: plotting and accuracy helpers.

def plot_curves(loss_history, accuracy_history, title='Training'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    ax1.plot(loss_history, linewidth=0.8, alpha=0.85)
    ax1.set_title(f'{title} — Loss')
    ax1.set_xlabel('Example (all epochs)')
    ax1.set_ylabel('Loss')
    ax2.plot(accuracy_history, linewidth=0.8, alpha=0.85, color='steelblue')
    ax2.set_title(f'{title} — Running accuracy')
    ax2.set_xlabel('Example (all epochs)')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

def accuracy_on(nn, dataset):
    correct = sum(1 for label, pixels in dataset if nn.predict(pixels) == label)
    return correct / len(dataset)

def show_misclassified(nn, test_data, n=12):
    errors = [(label, pixels) for label, pixels in test_data
              if nn.predict(pixels) != label]
    n = min(n, len(errors))
    if n == 0:
        print('No misclassified examples found.')
        return
    cols = 6
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 2.0))
    axes = axes.flat if rows > 1 else [axes] if cols == 1 else axes
    for i, ax in enumerate(axes):
        if i >= n:
            ax.axis('off')
            continue
        label, pixels = errors[i]
        pred = nn.predict(pixels)
        ax.imshow([pixels[j * 28:(j + 1) * 28] for j in range(28)], cmap='gray_r')
        ax.axis('off')
        ax.set_title(f'T:{label}  P:{pred}', fontsize=8, color='firebrick')
    plt.suptitle('Misclassified examples', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'{len(errors)} errors out of {len(test_data)} test examples.')

In [ ]:
# Load data and make an 80/20 split.
# Adjust max_rows to taste — 1000 trains in a reasonable time,
# more gives better curves.
all_data   = load_mnist_csv('mnist_train.csv', max_rows=1000)
split      = int(len(all_data) * 0.8)
train_data = all_data[:split]
test_data  = all_data[split:]

print(f'Training:  {len(train_data)} examples')
print(f'Test:      {len(test_data)} examples')

---

## The Experiment

We are going to train three networks on exactly the same data. The only thing that changes is their architecture — the number of layers and nodes.

Before we run anything, look at each architecture and use your `describe_network` function (or the `__str__` method on the class) to see the parameter count. This matters.

In [ ]:
# Three architectures.
tiny   = NeuralNetwork([784,  8,       10])   # very few nodes
medium = NeuralNetwork([784, 64, 32,   10])   # moderate
large  = NeuralNetwork([784, 512, 256, 10])   # many nodes

print('Tiny:')
print(tiny)
print()
print('Medium:')
print(medium)
print()
print('Large:')
print(large)

Look at those parameter counts. The large network has orders of magnitude more parameters than the tiny one, but they are all being trained on exactly the same number of examples. Write a brief prediction in the cell below: what do you expect will happen to each network?

In [ ]:
# Your prediction before training:
# Tiny   —
# Medium —
# Large  —

---

## Training All Three

In [ ]:
print('--- Tiny ---')
losses_tiny, accs_tiny = tiny.train(train_data, epochs=5, learning_rate=0.01)

In [ ]:
print('--- Medium ---')
losses_med, accs_med = medium.train(train_data, epochs=5, learning_rate=0.01)

In [ ]:
print('--- Large ---')
losses_large, accs_large = large.train(train_data, epochs=5, learning_rate=0.01)

---

## The Loss Curves

Plot the curves for each network. Look at the shape of the loss — not just the final value, but how it moves over the course of training.

In [ ]:
plot_curves(losses_tiny,  accs_tiny,  title='Tiny   [784 → 8 → 10]')

In [ ]:
plot_curves(losses_med,   accs_med,   title='Medium [784 → 64 → 32 → 10]')

In [ ]:
plot_curves(losses_large, accs_large, title='Large  [784 → 512 → 256 → 10]')

In calculus, the **rate of change** of a function describes how steeply it is rising or falling at any given point. If the loss were a smooth function `L(t)`, its derivative `L'(t)` would give the slope of the curve at each step. When the curve is falling steeply, `L'(t)` is a large negative number. When it has flattened out, `L'(t)` is near zero.

Answer the following, with reference to your curves:

1. Which network's loss falls the most steeply early on? Which flattens soonest?
2. If you were implementing **early stopping** — halting training automatically when improvement stalls — what quantity would you check at the end of each epoch, and what condition would trigger the stop?
3. A large negative derivative means the network is still learning rapidly. A derivative near zero means it has largely stopped improving. Given this, is a flat loss curve always a bad sign, or could it be a good sign in some circumstances?

In [ ]:
# Your answers:
#
# 1.
# 2.
# 3.

---

## Training vs Test Accuracy

Now let us measure how well each network performs on data it has *not* seen during training.

In [ ]:
results = {}
for name, nn in [('Tiny', tiny), ('Medium', medium), ('Large', large)]:
    tr = accuracy_on(nn, train_data)
    te = accuracy_on(nn, test_data)
    results[name] = (tr, te)
    print(f'{name:<8}  train: {tr:.2%}   test: {te:.2%}   gap: {tr - te:+.2%}')

The gap between training accuracy and test accuracy is the key number. Answer the following:

1. Which network has the largest gap? What does a large gap suggest about what that network has learned?
2. Which network has the smallest gap? Does that mean it is the best network, or is there another way to read the result?
3. A network that achieves high training accuracy but low test accuracy has effectively memorised its training data. We call this **overfitting**. In your own words: why does having many more parameters than training examples make overfitting more likely?

In [ ]:
# Your answers:
#
# 1.
# 2.
# 3.

---

## The Dimensionality of the Problem

Here is another way to think about what is happening.

The tiny network has very few parameters. In a sense, it has a very limited vocabulary for describing what it has learned — it can only represent simple, coarse patterns. It may not even be capable of learning a good solution, no matter how long you train it. This is called **underfitting**.

The large network has far more parameters than training examples. It has enough capacity to assign a unique combination of weights to every individual training image — it can memorise each one rather than finding a general pattern. When it sees a new image it has never encountered, it has no general knowledge to draw on.

Think of it this way: if a student studies for an exam by memorising exactly 40 past exam questions word for word, they will score perfectly on those 40 questions. But if the real exam has even slightly different phrasing or new examples of the same concept, that memorisation offers very little.

In [ ]:
# Look at the misclassified examples for each network.
print('Tiny — misclassified:')
show_misclassified(tiny, test_data)

In [ ]:
print('Medium — misclassified:')
show_misclassified(medium, test_data)

In [ ]:
print('Large — misclassified:')
show_misclassified(large, test_data)

Look at the images that each network gets wrong. Then answer:

1. The tiny network's errors likely include digits that look quite clear to you. What does this tell you about what it has (and has not) learned?
2. If any errors from the large network surprise you — cases where the image looks unambiguous but the prediction is wrong — what might explain that?
3. If you were to choose one of the three architectures to deploy in a real application, which would you choose, and what evidence from today's experiments supports that choice?

In [ ]:
# Your answers:
#
# 1.
# 2.
# 3.

---

That is the end of Day 3 and the end of the assessment. If you have time remaining, go back and tighten anything from Days 1 or 2 — particularly the class assembly and any test cells that are still incomplete.